In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

# -------- CHANGE THIS ONE LINE TO SELECT THE MODEL --------
MODEL_NAME = "Logistic Regression"
# ---------------------------------------------------------

RANDOM_STATE = 42
DATASET_PATH = "Datasets/heart_statlog_cleveland_hungary_final.csv"

# Load the original dataset directly.

data = pd.read_csv(DATASET_PATH)

TARGET_COLUMN = "target"

if TARGET_COLUMN not in data.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found. "
        f"Available columns: {list(data.columns)}"
    )

X = data.drop(columns=[TARGET_COLUMN]).copy()
y_original = data[TARGET_COLUMN].copy()

# Ensure features are numeric and handle missing values.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Encode target labels consistently as 0 and 1.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_original)

if len(label_encoder.classes_) != 2:
    raise ValueError(
        f"This code expects binary classification; found "
        f"{len(label_encoder.classes_)} target classes: "
        f"{list(label_encoder.classes_)}"
    )

print("Original target classes:", list(label_encoder.classes_))
print("Number of input features:", X.shape[1])

# Use the same stratified split for each baseline.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Fit preprocessing on training data only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------- MODEL CONFIGURATIONS ----------------

if MODEL_NAME == "Logistic Regression":

    hyperparameters = {
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 2000
    }

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            **hyperparameters,
            random_state=RANDOM_STATE
        )
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "SVM":

    hyperparameters = {
        "kernel": "rbf",
        "C": 1.0,
        "gamma": "scale",
        "probability": True
    }

    model = make_pipeline(
        StandardScaler(),
        SVC(**hyperparameters, random_state=RANDOM_STATE)
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "Random Forest":

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    }

    model = RandomForestClassifier(
        **hyperparameters,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "XGBoost":

    try:
        from xgboost import XGBClassifier
    except ImportError as exc:
        raise ImportError(
            "XGBoost is not installed. Run "
            "%pip install xgboost, then rerun this cell."
        ) from exc

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    }

    model = XGBClassifier(
        **hyperparameters,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

elif MODEL_NAME == "LSTM":

    # Standalone LSTM baseline, separate from the proposed model.
    LSTM_HIDDEN_UNITS = 64
    DROPOUT_RATE = 0.30
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 150

    hyperparameters = {
        "lstm_units": LSTM_HIDDEN_UNITS,
        "dropout": DROPOUT_RATE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "loss": "binary_crossentropy"
    }

    # LSTM input shape: (samples, features, 1)
    X_train_lstm = X_train_scaled[..., np.newaxis]
    X_test_lstm = X_test_scaled[..., np.newaxis]

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(X_train_lstm.shape[1], 1)
        ),
        tf.keras.layers.LSTM(LSTM_HIDDEN_UNITS),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_train_lstm,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    y_prob = model.predict(
        X_test_lstm, verbose=0
    ).ravel()

    y_pred = (y_prob >= 0.5).astype(int)

else:
    raise ValueError(
        "Unknown MODEL_NAME. Choose Logistic Regression, "
        "SVM, Random Forest, XGBoost, or LSTM."
    )

# ---------------- EVALUATION ----------------

results = {
    "Model": MODEL_NAME,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(
        y_test, y_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_pred, zero_division=0
    ),
    "F1 Score": f1_score(
        y_test, y_pred, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print("\nSelected model:", MODEL_NAME)
print("\nHyperparameters used:")
print(hyperparameters)

print("\nTest-set results:")
for metric, value in results.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=[str(c) for c in label_encoder.classes_],
    zero_division=0
))

# Save the current run's results and hyperparameters.
pd.DataFrame([results]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_results.csv",
    index=False
)

pd.DataFrame([{
    "Model": MODEL_NAME,
    **hyperparameters
}]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_hyperparameters.csv",
    index=False
)

print("\nResults and hyperparameters saved to CSV files.")


In [ ]:

# ============================================================
# BASELINE MODEL IMPLEMENTATION
# Change MODEL_NAME only to select a different model.
# Options: "Logistic Regression", "SVM", "Random Forest",
#          "XGBoost", "LSTM"
# ============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

# -------- CHANGE THIS ONE LINE TO SELECT THE MODEL --------
MODEL_NAME = "SVM"
# ---------------------------------------------------------

RANDOM_STATE = 42
DATASET_PATH = "Datasets/heart_statlog_cleveland_hungary_final.csv"

# Load the original dataset directly.
# This avoids accidentally using a DataFrame overwritten by plotting code.
data = pd.read_csv(DATASET_PATH)

TARGET_COLUMN = "target"

if TARGET_COLUMN not in data.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found. "
        f"Available columns: {list(data.columns)}"
    )

X = data.drop(columns=[TARGET_COLUMN]).copy()
y_original = data[TARGET_COLUMN].copy()

# Ensure features are numeric and handle missing values.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Encode target labels consistently as 0 and 1.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_original)

if len(label_encoder.classes_) != 2:
    raise ValueError(
        f"This code expects binary classification; found "
        f"{len(label_encoder.classes_)} target classes: "
        f"{list(label_encoder.classes_)}"
    )

print("Original target classes:", list(label_encoder.classes_))
print("Number of input features:", X.shape[1])

# Use the same stratified split for each baseline.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Fit preprocessing on training data only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------- MODEL CONFIGURATIONS ----------------

if MODEL_NAME == "Logistic Regression":

    hyperparameters = {
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 2000
    }

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            **hyperparameters,
            random_state=RANDOM_STATE
        )
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "SVM":

    hyperparameters = {
        "kernel": "rbf",
        "C": 1.0,
        "gamma": "scale",
        "probability": True
    }

    model = make_pipeline(
        StandardScaler(),
        SVC(**hyperparameters, random_state=RANDOM_STATE)
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "Random Forest":

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    }

    model = RandomForestClassifier(
        **hyperparameters,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "XGBoost":

    try:
        from xgboost import XGBClassifier
    except ImportError as exc:
        raise ImportError(
            "XGBoost is not installed. Run "
            "%pip install xgboost, then rerun this cell."
        ) from exc

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    }

    model = XGBClassifier(
        **hyperparameters,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

elif MODEL_NAME == "LSTM":

    # Standalone LSTM baseline, separate from the proposed model.
    LSTM_HIDDEN_UNITS = 64
    DROPOUT_RATE = 0.30
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 150

    hyperparameters = {
        "lstm_units": LSTM_HIDDEN_UNITS,
        "dropout": DROPOUT_RATE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "loss": "binary_crossentropy"
    }

    # LSTM input shape: (samples, features, 1)
    X_train_lstm = X_train_scaled[..., np.newaxis]
    X_test_lstm = X_test_scaled[..., np.newaxis]

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(X_train_lstm.shape[1], 1)
        ),
        tf.keras.layers.LSTM(LSTM_HIDDEN_UNITS),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_train_lstm,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    y_prob = model.predict(
        X_test_lstm, verbose=0
    ).ravel()

    y_pred = (y_prob >= 0.5).astype(int)

else:
    raise ValueError(
        "Unknown MODEL_NAME. Choose Logistic Regression, "
        "SVM, Random Forest, XGBoost, or LSTM."
    )

# ---------------- EVALUATION ----------------

results = {
    "Model": MODEL_NAME,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(
        y_test, y_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_pred, zero_division=0
    ),
    "F1 Score": f1_score(
        y_test, y_pred, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print("\nSelected model:", MODEL_NAME)
print("\nHyperparameters used:")
print(hyperparameters)

print("\nTest-set results:")
for metric, value in results.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=[str(c) for c in label_encoder.classes_],
    zero_division=0
))

# Save the current run's results and hyperparameters.
pd.DataFrame([results]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_results.csv",
    index=False
)

pd.DataFrame([{
    "Model": MODEL_NAME,
    **hyperparameters
}]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_hyperparameters.csv",
    index=False
)

print("\nResults and hyperparameters saved to CSV files.")


In [ ]:

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

# -------- CHANGE THIS ONE LINE TO SELECT THE MODEL --------
MODEL_NAME = "Random Forest"
# ---------------------------------------------------------

RANDOM_STATE = 42
DATASET_PATH = "Datasets/heart_statlog_cleveland_hungary_final.csv"

# Load the original dataset directly.
# This avoids accidentally using a DataFrame overwritten by plotting code.
data = pd.read_csv(DATASET_PATH)

TARGET_COLUMN = "target"

if TARGET_COLUMN not in data.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found. "
        f"Available columns: {list(data.columns)}"
    )

X = data.drop(columns=[TARGET_COLUMN]).copy()
y_original = data[TARGET_COLUMN].copy()

# Ensure features are numeric and handle missing values.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Encode target labels consistently as 0 and 1.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_original)

if len(label_encoder.classes_) != 2:
    raise ValueError(
        f"This code expects binary classification; found "
        f"{len(label_encoder.classes_)} target classes: "
        f"{list(label_encoder.classes_)}"
    )

print("Original target classes:", list(label_encoder.classes_))
print("Number of input features:", X.shape[1])

# Use the same stratified split for each baseline.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Fit preprocessing on training data only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------- MODEL CONFIGURATIONS ----------------

if MODEL_NAME == "Logistic Regression":

    hyperparameters = {
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 2000
    }

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            **hyperparameters,
            random_state=RANDOM_STATE
        )
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "SVM":

    hyperparameters = {
        "kernel": "rbf",
        "C": 1.0,
        "gamma": "scale",
        "probability": True
    }

    model = make_pipeline(
        StandardScaler(),
        SVC(**hyperparameters, random_state=RANDOM_STATE)
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "Random Forest":

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    }

    model = RandomForestClassifier(
        **hyperparameters,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "XGBoost":

    try:
        from xgboost import XGBClassifier
    except ImportError as exc:
        raise ImportError(
            "XGBoost is not installed. Run "
            "%pip install xgboost, then rerun this cell."
        ) from exc

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    }

    model = XGBClassifier(
        **hyperparameters,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

elif MODEL_NAME == "LSTM":

    # Standalone LSTM baseline, separate from the proposed model.
    LSTM_HIDDEN_UNITS = 64
    DROPOUT_RATE = 0.30
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 150

    hyperparameters = {
        "lstm_units": LSTM_HIDDEN_UNITS,
        "dropout": DROPOUT_RATE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "loss": "binary_crossentropy"
    }

    # LSTM input shape: (samples, features, 1)
    X_train_lstm = X_train_scaled[..., np.newaxis]
    X_test_lstm = X_test_scaled[..., np.newaxis]

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(X_train_lstm.shape[1], 1)
        ),
        tf.keras.layers.LSTM(LSTM_HIDDEN_UNITS),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_train_lstm,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    y_prob = model.predict(
        X_test_lstm, verbose=0
    ).ravel()

    y_pred = (y_prob >= 0.5).astype(int)

else:
    raise ValueError(
        "Unknown MODEL_NAME. Choose Logistic Regression, "
        "SVM, Random Forest, XGBoost, or LSTM."
    )

# ---------------- EVALUATION ----------------

results = {
    "Model": MODEL_NAME,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(
        y_test, y_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_pred, zero_division=0
    ),
    "F1 Score": f1_score(
        y_test, y_pred, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print("\nSelected model:", MODEL_NAME)
print("\nHyperparameters used:")
print(hyperparameters)

print("\nTest-set results:")
for metric, value in results.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=[str(c) for c in label_encoder.classes_],
    zero_division=0
))

# Save the current run's results and hyperparameters.
pd.DataFrame([results]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_results.csv",
    index=False
)

pd.DataFrame([{
    "Model": MODEL_NAME,
    **hyperparameters
}]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_hyperparameters.csv",
    index=False
)

print("\nResults and hyperparameters saved to CSV files.")


In [ ]:

# ============================================================
# BASELINE MODEL IMPLEMENTATION
# Change MODEL_NAME only to select a different model.
# Options: "Logistic Regression", "SVM", "Random Forest",
#          "XGBoost", "LSTM"
# ============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

# -------- CHANGE THIS ONE LINE TO SELECT THE MODEL --------
MODEL_NAME = "XGBoost"
# ---------------------------------------------------------

RANDOM_STATE = 42
DATASET_PATH = "Datasets/heart_statlog_cleveland_hungary_final.csv"

# Load the original dataset directly.
# This avoids accidentally using a DataFrame overwritten by plotting code.
data = pd.read_csv(DATASET_PATH)

TARGET_COLUMN = "target"

if TARGET_COLUMN not in data.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found. "
        f"Available columns: {list(data.columns)}"
    )

X = data.drop(columns=[TARGET_COLUMN]).copy()
y_original = data[TARGET_COLUMN].copy()

# Ensure features are numeric and handle missing values.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Encode target labels consistently as 0 and 1.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_original)

if len(label_encoder.classes_) != 2:
    raise ValueError(
        f"This code expects binary classification; found "
        f"{len(label_encoder.classes_)} target classes: "
        f"{list(label_encoder.classes_)}"
    )

print("Original target classes:", list(label_encoder.classes_))
print("Number of input features:", X.shape[1])

# Use the same stratified split for each baseline.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Fit preprocessing on training data only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------- MODEL CONFIGURATIONS ----------------

if MODEL_NAME == "Logistic Regression":

    hyperparameters = {
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 2000
    }

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            **hyperparameters,
            random_state=RANDOM_STATE
        )
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "SVM":

    hyperparameters = {
        "kernel": "rbf",
        "C": 1.0,
        "gamma": "scale",
        "probability": True
    }

    model = make_pipeline(
        StandardScaler(),
        SVC(**hyperparameters, random_state=RANDOM_STATE)
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "Random Forest":

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    }

    model = RandomForestClassifier(
        **hyperparameters,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "XGBoost":

    try:
        from xgboost import XGBClassifier
    except ImportError as exc:
        raise ImportError(
            "XGBoost is not installed. Run "
            "%pip install xgboost, then rerun this cell."
        ) from exc

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    }

    model = XGBClassifier(
        **hyperparameters,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

elif MODEL_NAME == "LSTM":

    # Standalone LSTM baseline, separate from the proposed model.
    LSTM_HIDDEN_UNITS = 64
    DROPOUT_RATE = 0.30
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 150

    hyperparameters = {
        "lstm_units": LSTM_HIDDEN_UNITS,
        "dropout": DROPOUT_RATE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "loss": "binary_crossentropy"
    }

    # LSTM input shape: (samples, features, 1)
    X_train_lstm = X_train_scaled[..., np.newaxis]
    X_test_lstm = X_test_scaled[..., np.newaxis]

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(X_train_lstm.shape[1], 1)
        ),
        tf.keras.layers.LSTM(LSTM_HIDDEN_UNITS),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_train_lstm,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    y_prob = model.predict(
        X_test_lstm, verbose=0
    ).ravel()

    y_pred = (y_prob >= 0.5).astype(int)

else:
    raise ValueError(
        "Unknown MODEL_NAME. Choose Logistic Regression, "
        "SVM, Random Forest, XGBoost, or LSTM."
    )

# ---------------- EVALUATION ----------------

results = {
    "Model": MODEL_NAME,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(
        y_test, y_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_pred, zero_division=0
    ),
    "F1 Score": f1_score(
        y_test, y_pred, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print("\nSelected model:", MODEL_NAME)
print("\nHyperparameters used:")
print(hyperparameters)

print("\nTest-set results:")
for metric, value in results.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=[str(c) for c in label_encoder.classes_],
    zero_division=0
))

# Save the current run's results and hyperparameters.
pd.DataFrame([results]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_results.csv",
    index=False
)

pd.DataFrame([{
    "Model": MODEL_NAME,
    **hyperparameters
}]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_hyperparameters.csv",
    index=False
)

print("\nResults and hyperparameters saved to CSV files.")


In [ ]:

# ============================================================
# BASELINE MODEL IMPLEMENTATION
# Change MODEL_NAME only to select a different model.
# Options: "Logistic Regression", "SVM", "Random Forest",
#          "XGBoost", "LSTM"
# ============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

# -------- CHANGE THIS ONE LINE TO SELECT THE MODEL --------
MODEL_NAME = "LSTM"
# ---------------------------------------------------------

RANDOM_STATE = 42
DATASET_PATH = "Datasets/heart_statlog_cleveland_hungary_final.csv"

# Load the original dataset directly.
# This avoids accidentally using a DataFrame overwritten by plotting code.
data = pd.read_csv(DATASET_PATH)

TARGET_COLUMN = "target"

if TARGET_COLUMN not in data.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found. "
        f"Available columns: {list(data.columns)}"
    )

X = data.drop(columns=[TARGET_COLUMN]).copy()
y_original = data[TARGET_COLUMN].copy()

# Ensure features are numeric and handle missing values.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Encode target labels consistently as 0 and 1.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_original)

if len(label_encoder.classes_) != 2:
    raise ValueError(
        f"This code expects binary classification; found "
        f"{len(label_encoder.classes_)} target classes: "
        f"{list(label_encoder.classes_)}"
    )

print("Original target classes:", list(label_encoder.classes_))
print("Number of input features:", X.shape[1])

# Use the same stratified split for each baseline.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Fit preprocessing on training data only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------- MODEL CONFIGURATIONS ----------------

if MODEL_NAME == "Logistic Regression":

    hyperparameters = {
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 2000
    }

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            **hyperparameters,
            random_state=RANDOM_STATE
        )
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "SVM":

    hyperparameters = {
        "kernel": "rbf",
        "C": 1.0,
        "gamma": "scale",
        "probability": True
    }

    model = make_pipeline(
        StandardScaler(),
        SVC(**hyperparameters, random_state=RANDOM_STATE)
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "Random Forest":

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    }

    model = RandomForestClassifier(
        **hyperparameters,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

elif MODEL_NAME == "XGBoost":

    try:
        from xgboost import XGBClassifier
    except ImportError as exc:
        raise ImportError(
            "XGBoost is not installed. Run "
            "%pip install xgboost, then rerun this cell."
        ) from exc

    hyperparameters = {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    }

    model = XGBClassifier(
        **hyperparameters,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

elif MODEL_NAME == "LSTM":

    # Standalone LSTM baseline, separate from the proposed model.
    LSTM_HIDDEN_UNITS = 64
    DROPOUT_RATE = 0.30
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 150

    hyperparameters = {
        "lstm_units": LSTM_HIDDEN_UNITS,
        "dropout": DROPOUT_RATE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "loss": "binary_crossentropy"
    }

    # LSTM input shape: (samples, features, 1)
    X_train_lstm = X_train_scaled[..., np.newaxis]
    X_test_lstm = X_test_scaled[..., np.newaxis]

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(X_train_lstm.shape[1], 1)
        ),
        tf.keras.layers.LSTM(LSTM_HIDDEN_UNITS),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_train_lstm,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    y_prob = model.predict(
        X_test_lstm, verbose=0
    ).ravel()

    y_pred = (y_prob >= 0.5).astype(int)

else:
    raise ValueError(
        "Unknown MODEL_NAME. Choose Logistic Regression, "
        "SVM, Random Forest, XGBoost, or LSTM."
    )

# ---------------- EVALUATION ----------------

results = {
    "Model": MODEL_NAME,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(
        y_test, y_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_pred, zero_division=0
    ),
    "F1 Score": f1_score(
        y_test, y_pred, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print("\nSelected model:", MODEL_NAME)
print("\nHyperparameters used:")
print(hyperparameters)

print("\nTest-set results:")
for metric, value in results.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=[str(c) for c in label_encoder.classes_],
    zero_division=0
))

# Save the current run's results and hyperparameters.
pd.DataFrame([results]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_results.csv",
    index=False
)

pd.DataFrame([{
    "Model": MODEL_NAME,
    **hyperparameters
}]).to_csv(
    f"baseline_{MODEL_NAME.lower().replace(' ', '_')}_hyperparameters.csv",
    index=False
)

print("\nResults and hyperparameters saved to CSV files.")
